In [1]:
import re
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime

In [2]:
# Logging function to display messages and append them to logfile.txt
def log(message):
    
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    log_entry = f"[{timestamp}] {message}"
    
    print(log_entry)
    
    with open("logfile.txt", "a", encoding="utf-8") as f:
        f.write(log_entry + "\n")

In [3]:
log("Jupyter Notebook Environment Initialized.")

[2026-09-11 14:40:57] Jupyter Notebook Environment Initialized.


In [4]:
# Stage 1: Read and acquire raw dataset
def extract(file_path):
    log("=== STAGE 1: EXTRACTION ===")
    try:
        df = pd.read_csv(file_path)
        log(f"Data extracted successfully. Shape: {df.shape[0]} rows, {df.shape[1]} columns.")
        return df
    except Exception as e:
        log(f"Extraction Error: {e}")
        raise e

In [5]:
def clean_price_inr(val):
    """Standardize INR text while preserving unavailable-price labels."""
    if pd.isna(val):
        return np.nan

    value = str(val).strip()
    lowered = value.lower()
    unavailable = ('call for price', 'price on request', 'contact us')
    if any(phrase in lowered for phrase in unavailable):
        return "Call for Price"

    if len(value) > 30 and not re.search(r'(lac|cr)', lowered):
        return np.nan

    match = re.search(r'([\d]+(?:\.\d+)?)\s*(lac|cr)\b', lowered)
    if match:
        multiplier = 100000 if match.group(2) == 'lac' else 10000000
        return float(match.group(1)) * multiplier

    cleaned = re.sub(r'[^\d.]', '', value)
    try:
        return float(cleaned) if cleaned else np.nan
    except ValueError:
        return np.nan
    
def price_to_numeric(val):
    """Return numeric INR for calculations and NaN for Call for Price."""
    if pd.isna(val) or str(val).strip().lower() == 'call for price':
        return np.nan
    try:
        return float(val)
    except (TypeError, ValueError):
        return np.nan

In [6]:
def extract_numeric_area(val):
    """
    Extracts raw numeric sqft value from area text, strictly ignoring descriptive text.
    """
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip()
    
    if len(val_str) > 25 or (re.search(r'[a-zA-Z]{4,}', val_str) and not re.search(r'sqft', val_str, re.IGNORECASE)):
        return np.nan
        
    match = re.search(r'([\d\.]+)', val_str)
    if match:
        try:
            val_num = float(match.group(1))
            return val_num if val_num > 0 else np.nan
        except ValueError:
            return np.nan
    return np.nan

In [7]:
def transform(df):
    """Clean, standardize, derive missing prices, and quality-flag the dataset."""
    log("=== STAGE 2: TRANSFORMATION & CLEANING (INR STANDARDIZATION) ===")
    df_clean = df.copy()

    initial_rows = len(df_clean)
    df_clean = df_clean.drop_duplicates().reset_index(drop=True)
    log(f"Removed {initial_rows - len(df_clean)} duplicate rows.")

    amount_col = 'Amount(in rupees)'
    unit_price_col = 'Price (in rupees)'
    df_clean[amount_col] = df_clean[amount_col].apply(clean_price_inr).astype('object')
    df_clean[unit_price_col] = df_clean[unit_price_col].apply(clean_price_inr).astype('object')

    log("Merging Carpet Area & Super Area into 'Area_sqft'...")
    carpet_num = (
        df_clean['Carpet Area'].apply(extract_numeric_area)
        if 'Carpet Area' in df_clean.columns
        else pd.Series(np.nan, index=df_clean.index)
    )
    super_num = (
        df_clean['Super Area'].apply(extract_numeric_area)
        if 'Super Area' in df_clean.columns
        else pd.Series(np.nan, index=df_clean.index)
    )
    df_clean['Area_sqft'] = carpet_num.fillna(super_num)

    # Keep numeric values temporary so the output schema stays focused on source fields.
    amount_numeric = df_clean[amount_col].apply(price_to_numeric)
    unit_price_numeric = df_clean[unit_price_col].apply(price_to_numeric)

    can_derive_amount = (
        amount_numeric.isna()
        & df_clean[amount_col].ne('Call for Price')
        & unit_price_numeric.notna()
        & df_clean['Area_sqft'].notna()
    )
    derived_amount = unit_price_numeric * df_clean['Area_sqft']
    amount_numeric = amount_numeric.mask(can_derive_amount, derived_amount)
    df_clean.loc[can_derive_amount, amount_col] = derived_amount[can_derive_amount]

    can_derive_unit_price = (
        unit_price_numeric.isna()
        & df_clean[unit_price_col].ne('Call for Price')
        & amount_numeric.notna()
        & df_clean['Area_sqft'].gt(0)
    )
    derived_unit_price = amount_numeric / df_clean['Area_sqft']
    unit_price_numeric = unit_price_numeric.mask(can_derive_unit_price, derived_unit_price)
    df_clean.loc[can_derive_unit_price, unit_price_col] = derived_unit_price[can_derive_unit_price]

    # Replace unresolved visible price blanks with the requested label.
    df_clean.loc[df_clean[amount_col].isna(), amount_col] = 'Call for Price'
    df_clean.loc[df_clean[unit_price_col].isna(), unit_price_col] = 'Call for Price'

    cols_to_drop_area = [c for c in ['Carpet Area', 'Super Area'] if c in df_clean.columns]
    if cols_to_drop_area:
        df_clean.drop(columns=cols_to_drop_area, inplace=True)

    log("Standardizing categorical fields...")
    categorical_defaults = {
        'facing': 'Unspecified',
        'overlooking': 'Not Specified',
        'Society': 'Independent / None',
        'Balcony': '0',
        'Car Parking': 'No Parking',
        'Ownership': 'Unknown Ownership',
        'location': 'Unknown Location',
        'Status': 'Unknown Status',
        'Transaction': 'Unknown Transaction',
        'Furnishing': 'Unknown Furnishing'
    }
    for column, default in categorical_defaults.items():
        if column in df_clean.columns:
            cleaned = df_clean[column].fillna(default).astype(str).str.strip()
            if column in {'facing', 'Ownership', 'location', 'Status', 'Transaction', 'Furnishing'}:
                cleaned = cleaned.str.title()
            df_clean[column] = cleaned.replace('', default)

    empty_columns = [column for column in df_clean.columns if df_clean[column].isna().all()]
    if empty_columns:
        log(f"Dropping empty columns: {empty_columns}")
        df_clean.drop(columns=empty_columns, inplace=True)

    has_amount = amount_numeric.notna() & (amount_numeric > 0)
    has_area = df_clean['Area_sqft'].notna() & (df_clean['Area_sqft'] > 0)
    df_clean['is_invalid'] = ~(has_amount & has_area)
    df_clean['price_status'] = np.where(
        df_clean[amount_col].eq('Call for Price'),
        'Call for Price',
        np.where(amount_numeric.isna(), 'Missing', 'Available')
    )
    log(
        f"Flagged {int(df_clean['is_invalid'].sum()):,} invalid records; "
        f"preserved {int(df_clean['price_status'].eq('Call for Price').sum()):,} Call for Price records."
    )
    return df_clean

In [8]:
def price_to_float(val):
    """
    Helper function to parse numeric values from cleaned INR strings or floats
    while safely ignoring text like 'call for price'.
    """
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    
    cleaned = re.sub(r'[^\d.]', '', str(val))
    try:
        return float(cleaned) if cleaned else np.nan
    except ValueError:
        return np.nan

In [9]:
def analyze(df):
    """Create location summaries using temporary numeric values without adding output columns."""
    log("=== STAGE 3: ANALYSIS & SUMMARY CREATION ===")
    try:
        valid_df = df.loc[~df['is_invalid']].copy() if 'is_invalid' in df.columns else df.copy()
        amount_col = 'Amount(in rupees)'
        unit_price_col = 'Price (in rupees)'

        if amount_col not in valid_df.columns or 'location' not in valid_df.columns:
            summary_df = pd.DataFrame({'status': ['Required price or location column not found']})
        else:
            numeric_amount = valid_df[amount_col].apply(price_to_numeric)
            numeric_unit_price = valid_df[unit_price_col].apply(price_to_numeric)
            analysis_df = valid_df[['location', 'Area_sqft']].copy()
            analysis_df['amount'] = numeric_amount
            analysis_df['unit_price'] = numeric_unit_price

            summary_df = (
                analysis_df.groupby('location', dropna=False)
                .agg(
                    total_properties=('amount', 'count'),
                    total_value_inr=('amount', 'sum'),
                    avg_price_inr=('amount', 'mean'),
                    median_price_inr=('amount', 'median'),
                    avg_area_sqft=('Area_sqft', 'mean'),
                    median_price_per_sqft_inr=('unit_price', 'median')
                )
                .reset_index()
                .sort_values(['total_properties', 'median_price_inr'], ascending=[False, False])
            )

        log("Summary aggregation table generated successfully.")
        return summary_df
    except Exception as e:
        log(f"Analysis Error: {e}")
        raise

In [10]:
# Stage 4: Persist output datasets into CSV files and SQLite database
def load(df, summary_df, csv_output, summary_output, db_output):
    from pathlib import Path

    log("=== STAGE 4: LOADING ===")

    def save_csv(dataframe, output_path, label):
        try:
            dataframe.to_csv(output_path, index=False)
            return output_path
        except PermissionError:
            primary = Path(output_path)
            fallback = primary.with_name(f"{primary.stem}_latest{primary.suffix}")
            dataframe.to_csv(fallback, index=False)
            log(f"Primary {label} is locked; saved fallback output to: {fallback}")
            return str(fallback)

    try:
        clean_path = save_csv(df, csv_output, "clean dataset")
        summary_path = save_csv(summary_df, summary_output, "summary dataset")
        log(f"Saved clean dataset to: {clean_path}")
        log(f"Saved summary dataset to: {summary_path}")

        try:
            conn = sqlite3.connect(db_output)
        except PermissionError:
            fallback_db = Path(db_output).with_name(f"{Path(db_output).stem}_latest.db")
            conn = sqlite3.connect(fallback_db)
            log(f"Primary database is locked; saved fallback output to: {fallback_db}")

        with conn:
            df.to_sql('cleaned_house_prices', conn, if_exists='replace', index=False)
            summary_df.to_sql('summary_house_prices', conn, if_exists='replace', index=False)
        log("Saved cleaned tables into SQLite DB successfully.")
    except Exception as e:
        log(f"Loading Error: {e}")
        raise

In [11]:
def main():
    log("Starting Full Data Engineering Pipeline...")
    
    input_csv = "data/raw/house_prices.csv" 
    cleaned_csv = "data/output/cleaned_house_prices.csv"
    summary_csv = "data/output/summary_house_prices.csv"
    db_file = "data/output/house_prices.db"
    
    try:
        # 1. Extract
        raw_data = extract(input_csv)
        
        # 2. Transform
        clean_data = transform(raw_data)
        
        # 3. Analyze
        summary_data = analyze(clean_data)
        
        # 4. Save/Load to Files
        load(clean_data, summary_data, cleaned_csv, summary_csv, db_file)
        
        print("\n" + "="*50)
        print("✅ VERIFICATION SAMPLE FROM CLEANED DATA:")
        print("="*50)
        # Print sample to confirm USD and merged Area in console
        sample_cols = [c for c in ['Price (in rupees)', 'Amount(in rupees)', 'Area_sqft', 'facing'] if c in clean_data.columns]
        print(clean_data[sample_cols].head(5))
        print("="*50 + "\n")
        
        log("ETL Pipeline Finished Successfully!")
        
    except Exception as e:
        log(f"Pipeline execution failed: {e}")
        raise e

# Run pipeline
if __name__ == "__main__":
    main()

[2026-09-11 17:23:56] Starting Full Data Engineering Pipeline...
[2026-09-11 17:23:56] === STAGE 1: EXTRACTION ===
[2026-09-11 17:23:57] Data extracted successfully. Shape: 187531 rows, 21 columns.
[2026-09-11 17:23:57] === STAGE 2: TRANSFORMATION & CLEANING (INR STANDARDIZATION) ===
[2026-09-11 17:23:58] Removed 0 duplicate rows.
[2026-09-11 17:23:59] Merging Carpet Area & Super Area into 'Area_sqft'...
[2026-09-11 17:23:59] Standardizing categorical fields...
[2026-09-11 17:24:00] Dropping empty columns: ['Dimensions', 'Plot Area']
[2026-09-11 17:24:00] Flagged 17,920 invalid records; preserved 9,684 Call for Price records.
[2026-09-11 17:24:00] === STAGE 3: ANALYSIS & SUMMARY CREATION ===
[2026-09-11 17:24:00] Summary aggregation table generated successfully.
[2026-09-11 17:24:00] === STAGE 4: LOADING ===
[2026-09-11 17:24:02] Saved clean dataset to: data/output/cleaned_house_prices.csv
[2026-09-11 17:24:02] Saved summary dataset to: data/output/summary_house_prices.csv
[2026-09-11 

In [ ]:
# Extract & Transform the raw dataset to create clean_data
raw_data = extract("data/raw/house_prices.csv")
clean_data = transform(raw_data)

# Focused ETL quality checks: keep the output schema free of temporary price columns.
for temporary_column in {
    'amount_inr_numeric',
    'price_per_sqft_inr_numeric',
    'price_per_sqft_inr'
}:
    assert temporary_column not in clean_data.columns
assert clean_data['Amount(in rupees)'].notna().all()
assert clean_data['Price (in rupees)'].notna().all()
assert clean_data['is_invalid'].dtype == bool
assert clean_data.loc[clean_data['Index'] == 0, 'Amount(in rupees)'].iloc[0] == 4200000.0
call_for_price_count = clean_data['price_status'].eq('Call for Price').sum()
print(f"Validated {len(clean_data):,} rows")
print(f"Call for Price records preserved: {call_for_price_count:,}")
print(f"Final output columns: {', '.join(clean_data.columns)}")

In [ ]:
# Create a filtered DataFrame containing only valid records for analytical calculations
# Ensure clean_data exists in the notebook's global scope
if 'clean_data' not in globals():
    raw_data = extract("data/raw/house_prices.csv")
    clean_data = transform(raw_data)

# Create a filtered DataFrame containing only valid records
valid_data = (
    clean_data[clean_data['is_invalid'] == False].copy()
    if 'is_invalid' in clean_data.columns
    else clean_data.copy()
)
print(f"Total Valid Records for Analysis: {len(valid_data):,} rows")

In [ ]:
import re
import numpy as np
import pandas as pd

raw_data = extract("data/raw/house_prices.csv")
clean_data = transform(raw_data)

def usd_to_float(val):
    if pd.isna(val): 
        return np.nan
    cleaned = re.sub(r'[^\d.]', '', str(val))
    try: 
        return float(cleaned)
    except: 
        return np.nan

valid_data = clean_data[clean_data['is_invalid'] == False].copy()
valid_data['price_usd_num'] = valid_data['Price (in rupees)'].apply(usd_to_float)
valid_data['price_per_sqft_usd'] = valid_data['price_usd_num'] / valid_data['Area_sqft']

print(f"Data ready for analysis. Valid records: {len(valid_data):,} rows")

In [ ]:
# Q1: Which locations have the highest median/average property price?

loc_summary = valid_data.groupby('location')['price_usd_num'].agg(['count', 'mean', 'median']).reset_index()

# Filter locations with at least 5 records to ensure statistical accuracy
top_avg_locs = loc_summary[loc_summary['count'] >= 5].sort_values(by='mean', ascending=False).head(5)
top_med_locs = loc_summary[loc_summary['count'] >= 5].sort_values(by='median', ascending=False).head(5)

print("="*65)
print("📊 Q1: TOP LOCATIONS BY AVERAGE & MEDIAN PROPERTY PRICE (USD)")
print("="*65)

print("\n1. Top 5 Locations by Average Price ($):")
for idx, row in top_avg_locs.reset_index(drop=True).iterrows():
    print(f"  {idx+1}. {row['location']:<30} | Avg Price: ${row['mean']:,.2f}")

print("\n2. Top 5 Locations by Median Price ($):")
for idx, row in top_med_locs.reset_index(drop=True).iterrows():
    print(f"  {idx+1}. {row['location']:<30} | Median Price: ${row['median']:,.2f}")

In [ ]:
# Q2: Which property characteristics are associated with higher prices in descriptive analysis?

print("="*65)
print("📊 Q2: PRICE BREAKDOWN BY PROPERTY CHARACTERISTICS")
print("="*65)

# A. Price vs Furnishing Status
if 'Furnishing' in valid_data.columns:
    # Use the actual numeric price column present in valid_data
    price_col = (
        'price_usd_num'
        if 'price_usd_num' in valid_data.columns
        else 'price_rupees_clean'
        if 'price_rupees_clean' in valid_data.columns
        else None
    )

    if price_col is not None:
        furn_price = (
            valid_data.dropna(subset=['Furnishing', price_col])
            .groupby('Furnishing')[price_col]
            .agg(['count', 'median'])
            .sort_values(by='median', ascending=False)
        )
    else:
        print("No usable price column found for furnishing analysis.")

    # Optional: also fix the Bathroom block in the same cell
    if 'Bathroom' in valid_data.columns and price_col is not None:
        bath_price = (
            valid_data.dropna(subset=['Bathroom', price_col])
            .groupby('Bathroom')[price_col]
            .agg(['count', 'median'])
            .sort_values(by='median', ascending=False)
            .head(5)
        )
        print("\n2. Median Price by Bathroom Count:")
        for bath, row in bath_price.iterrows():
            print(f"  • {bath} Bathrooms{' ':<10} | Median Price: ${row['median']:,.2f}")
    print("\n1. Median Price by Furnishing Status:")
    for status, row in furn_price.iterrows():
        print(f"  • {status:<20} | Median Price: ₹{row['median']:,.2f} ({row['count']:,} properties)")

# B. Price vs Bathrooms
if 'Bathroom' in valid_data.columns:
    if price_col is not None:
        bath_price = (
            valid_data.dropna(subset=['Bathroom', price_col])
            .groupby('Bathroom')[price_col]
            .agg(['count', 'median'])
            .sort_values(by='median', ascending=False)
            .head(5)
        )
        print("\n2. Median Price by Bathroom Count:")
        for bath, row in bath_price.iterrows():
            print(f"  • {bath} Bathrooms{' ':<10} | Median Price: ${row['median']:,.2f}")
    else:
        print("No usable price column found for bathroom analysis.")
    print("\n2. Median Price by Bathroom Count:")
    for bath, row in bath_price.iterrows():
        print(f"  • {bath} Bathrooms{' ':<10} | Median Price: ₹{row['median']:,.2f}")

In [ ]:
# Q3: Which locations have the highest price per area?

print("="*65)
print("📊 Q3: TOP LOCATIONS BY PRICE PER SQUARE FOOT")
print("="*65)

if 'price_per_carpet_area' in valid_data.columns:
    top_ppa = valid_data.groupby('location')['price_per_carpet_area'].agg(['count', 'median']).reset_index()
    # Filter for locations with at least 5 properties
    top_ppa = top_ppa[top_ppa['count'] >= 5].sort_values(by='median', ascending=False).head(5)
    
    print("\nTop 5 Locations with Highest Median Price per SqFt:")
    for idx, row in top_ppa.reset_index(drop=True).iterrows():
        print(f"  {idx+1}. {row['location']:<30} | Price/SqFt: ₹{row['median']:,.2f}")
else:
    print("Column 'price_per_carpet_area' was not found.")

In [ ]:
# Q4: How many records have missing critical price/area values?

print("="*65)
print("📊 Q4: MISSING CRITICAL VALUES AUDIT (PRICE & AREA)")
print("="*65)

missing_prices = clean_data['cleaned_price'].isna().sum() if 'cleaned_price' in clean_data.columns else clean_data['Price (in rupees)'].isna().sum()
missing_areas = (
    clean_data['Area_sqft'].isna().sum()
    if 'Area_sqft' in clean_data.columns
    else clean_data['final_area_sqft'].isna().sum()
    if 'final_area_sqft' in clean_data.columns
    else clean_data['Carpet Area'].isna().sum()
    if 'Carpet Area' in clean_data.columns
    else 0
)

if 'cleaned_price' in clean_data.columns and 'final_area_sqft' in clean_data.columns:
    both_missing = clean_data[(clean_data['cleaned_price'].isna()) & (clean_data['final_area_sqft'].isna())].shape[0]
else:
    both_missing = 0

total_records = len(clean_data)

print(f"\nTotal Raw Records Evaluated: {total_records:,}\n")
print(f"  • Missing Clean Price Values : {missing_prices:,} records ({(missing_prices/total_records)*100:.2f}%)")
print(f"  • Missing Clean Area Values  : {missing_areas:,} records ({(missing_areas/total_records)*100:.2f}%)")
print(f"  • Both Price & Area Missing  : {both_missing:,} records ({(both_missing/total_records)*100:.2f}%)")

In [ ]:
# Q5: Which furnishing/status categories are most common?

print("="*65)
print("📊 Q5: MOST COMMON PROPERTY CATEGORIES DISTRIBUTION")
print("="*65)

total_records = len(clean_data)

# A. Furnishing Category Distribution
if 'Furnishing' in clean_data.columns:
    top_furn = clean_data['Furnishing'].value_counts()
    print("\n1. Furnishing Category Breakdown:")
    for category, count in top_furn.items():
        print(f"  • {category:<20}: {count:,} properties ({(count/total_records)*100:.1f}%)")

# B. Status Category Distribution
if 'Status' in clean_data.columns:
    top_status = clean_data['Status'].value_counts()
    print("\n2. Property Status Breakdown:")
    for category, count in top_status.items():
        print(f"  • {category:<20}: {count:,} properties ({(count/total_records)*100:.1f}%)")

In [ ]:
# Q6: How many duplicate/invalid records are found?

print("="*65)
print("📊 Q6: DATA QUALITY AUDIT - DUPLICATES & INVALID RECORDS")
print("="*65)

total_records = len(clean_data)
invalid_count = clean_data['is_invalid'].sum() if 'is_invalid' in clean_data.columns else 0
valid_count = total_records - invalid_count

print(f"\n1. Duplicate Records  : Removed during extract/transform stage.")
print(f"2. Flagged Invalid Records (`is_invalid = True` ) : {invalid_count:,} records ({(invalid_count/total_records)*100:.2f}%)")
print(f"3. Valid Retained Records  (`is_invalid = False`) : {valid_count:,} records ({(valid_count/total_records)*100:.2f}%)")
print("\nNote: Invalid records (zero or negative prices/areas) were flagged rather than silently deleted.")